# Judge Model Training (Regression)

This is the main notebook used to train the DistilBERT-based judge model. The model learns to predict rating intensity for the target rating groups `1, 2, 3, 4, 7, 8, 9, 10` and is later used to score generated representative reviews.

**Input:** `data/imdb_sup.csv`

**Outputs:** `artifacts/judge_regression_checkpoints/`, `artifacts/judge_regression_model/`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertConfig,
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

PROJECT_ROOT = next(
    (path.resolve() for path in (Path.cwd(), Path.cwd().parent) if (path / "data" / "imdb_sup.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/imdb_sup.csv. Run the notebook from the repository root or notebooks/ directory."
    )

DATA_PATH = PROJECT_ROOT / 'data' / 'imdb_sup.csv'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
CHECKPOINT_DIR = ARTIFACTS_DIR / 'judge_regression_checkpoints'
MODEL_DIR = ARTIFACTS_DIR / 'judge_regression_model'
LOG_DIR = ARTIFACTS_DIR / 'logs'

for path in (ARTIFACTS_DIR, CHECKPOINT_DIR, MODEL_DIR, LOG_DIR):
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(DATA_PATH, on_bad_lines='skip', engine='python')

target_ratings = [1, 2, 3, 4, 7, 8, 9, 10]
df = df[df['Rating'].isin(target_ratings)].copy()
df['label'] = df['Rating'].astype(float)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Review'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
)

print(f"Loaded {len(df):,} filtered reviews from {DATA_PATH}")
print(f"Training examples: {len(train_texts):,}")
print(f"Validation examples: {len(val_texts):,}")


In [ ]:
config = DistilBertConfig.from_pretrained(
    'distilbert-base-uncased',
    num_labels=1,
    dropout=0.3,
    attention_dropout=0.3,
)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    config=config,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model is running on: {device}")


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    mse = mean_squared_error(labels, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(labels, predictions)
    return {'mse': mse, 'rmse': rmse, 'mae': mae}

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)

train_dict = {'text': train_texts, 'labels': train_labels}
val_dict = {'text': val_texts, 'labels': val_labels}

raw_train_dataset = Dataset.from_dict(train_dict)
raw_val_dataset = Dataset.from_dict(val_dict)

train_dataset = raw_train_dataset.map(tokenize_function, batched=True)
val_dataset = raw_val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.3,
    warmup_steps=500,
    logging_dir=str(LOG_DIR),
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)


In [ ]:
resume_from_checkpoint = True if any(CHECKPOINT_DIR.glob('checkpoint-*')) else None
if resume_from_checkpoint:
    print(f"Resuming from checkpoint in {CHECKPOINT_DIR}")
else:
    print('Starting training from scratch')

trainer.train(resume_from_checkpoint=resume_from_checkpoint)


In [ ]:
print('Generating predictions...')
predictions_output = trainer.predict(val_dataset)
raw_preds = predictions_output.predictions.flatten()
true_vals = predictions_output.label_ids.astype(int)

valid_ratings = np.array([1, 2, 3, 4, 7, 8, 9, 10])

def round_to_nearest_valid(pred):
    idx = (np.abs(valid_ratings - pred)).argmin()
    return int(valid_ratings[idx])

rounded_preds = [round_to_nearest_valid(pred) for pred in raw_preds]

final_accuracy = accuracy_score(true_vals, rounded_preds)
print(f"\nFinal Accuracy (after rounding): {final_accuracy:.2%}")
print('-' * 30)
print('Classification Report:')
print(classification_report(true_vals, rounded_preds, zero_division=0))

cm = confusion_matrix(true_vals, rounded_preds, labels=valid_ratings)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=valid_ratings)

plt.figure(figsize=(12, 12))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title(f'Regression Results: Rounded to Nearest Rating\nAccuracy: {final_accuracy:.2%}')
plt.show()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f'Model saved to {MODEL_DIR}')


## Optional Spot Check

Use the saved regression judge to score individual review snippets after training finishes.


In [ ]:
print('Loading model...')
loaded_model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)
loaded_tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model.to(device)
loaded_model.eval()

def predict_rating(text):
    inputs = loaded_tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    prediction = outputs.logits.item()
    valid_ratings = np.array([1, 2, 3, 4, 7, 8, 9, 10])
    idx = (np.abs(valid_ratings - prediction)).argmin()
    final_rating = int(valid_ratings[idx])
    return prediction, final_rating

sample_review = 'This movie was absolutely terrible, I wasted my time.'
raw_score, rounded_score = predict_rating(sample_review)

print(f'Review: {sample_review}')
print(f'Raw Prediction: {raw_score:.4f}')
print(f'Final Rating: {rounded_score} star(s)')

print('-' * 30)
user_review = ''
while user_review != 'exit':
    user_review = input("Enter a review (or 'exit' to stop): ")
    if user_review != 'exit':
        raw_score, rounded_score = predict_rating(user_review)
        print(f'Review: {user_review}')
        print(f'Final Rating: {rounded_score} star(s)')
